# Day 6: Mini Integration project | DES + ML
## Objective
The objective of this project is to integrate discrete-event simulation (DES) with machine learning (ML) for a mining truck-shovel system. A SimPy-based DES model is used to generate synthetic operational data such as queue time, loading time, travel time, and cycle time. This simulated data is then used to train a machine-learning regression model to predict truck cycle time.

## Inputs
- Number of trucks
- Loading time
- Travel time
- Dumping time
- Return time
- Queue time

## Output
- Predicted truck cycle time
- Model performance metrics (MAE and R²)

## What Was Changed
The simulation-generated operational data was integrated with a machine-learning workflow to demonstrate how predictive analytics can support intelligent fleet-management systems in mining operations.

In [5]:
import simpy
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score

# running a DES and storing the synthetic resuts in a Dataframe

RANDOM_SEED = 42
SIM_TIME = 8 * 60
N_TRUCKS = 6

random.seed(RANDOM_SEED)

records = []

def truck_cycle(env, truck_id, shovel):

    cycle = 0

    while env.now < SIM_TIME:

        cycle += 1

        arrival_time = env.now

        with shovel.request() as request:

            yield request

            queue_time = env.now - arrival_time

            loading_time = random.uniform(4, 8)
            yield env.timeout(loading_time)

            loaded_travel = random.uniform(10, 18)
            yield env.timeout(loaded_travel)

            dumping_time = random.uniform(2, 4)
            yield env.timeout(dumping_time)

            empty_return = random.uniform(8, 14)
            yield env.timeout(empty_return)

            # some random noise added in cycle time, to represent real world
            cycle_time = env.now - arrival_time + random.uniform(-4,4)

            records.append({
                "queue_time": queue_time,
                "loading_time": loading_time,
                "loaded_travel": loaded_travel,
                "dumping_time": dumping_time,
                "empty_return": empty_return,
                "cycle_time": cycle_time
            })

env = simpy.Environment()

shovel = simpy.Resource(env, capacity=1)

for i in range(N_TRUCKS):
    env.process(truck_cycle(env, f"Truck_{i+1}", shovel))

env.run(until=SIM_TIME)

df = pd.DataFrame(records)

print(df.head())

   queue_time  loading_time  loaded_travel  dumping_time  empty_return  \
0    0.000000      6.557707      10.200086      2.550059      9.339264   
1   28.647116      6.706798      17.137437      2.173878     10.531531   
2   65.196759      4.874552      14.042842      2.053072      9.193026   
3   95.360251      6.179766      11.763525      3.178531     12.856583   
4  129.338656      7.223277      15.585115      2.680501      8.932877   

   cycle_time  
0   30.538886  
1   61.435137  
2   96.559327  
3  125.390647  
4  167.418131  


 In the following part, we distinguish series into features and target variable, then split the data into test and train, and then train a linear regression model to predict the cycle time of test data. This is an example of supervised regression. The MAE and R2 scores are generated to judge the model

In [6]:
#We will not use all the features, because then the model will be deterministic and have nothing to predict
# we use use queue time, loaded travel time, and empty return time only.
X = df[[
    "queue_time",
    "loaded_travel",
    "empty_return"
]]

y = df["cycle_time"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42
)

model = LinearRegression()

model.fit(X_train, y_train)

pred = model.predict(X_test)

print("MAE:", round(mean_absolute_error(y_test, pred), 2))
print("R2 :", round(r2_score(y_test, pred), 3))

MAE: 3.56
R2 : 0.996


## Inference

Initially, the model produced perfect predictions because the target variable (`cycle_time`) was directly dependent on all input features, making the relationship deterministic. To create a more realistic prediction problem, random operational noise was added to the cycle time, and only a subset of features (`queue_time`, `loaded_travel`, and `empty_return`) was used for training. After these modifications, the model achieved an MAE of 3.56 and an R² score of 0.996, showing strong predictive capability while incorporating realistic uncertainty and variability.